# Managed system

## Overview

In this notebook we are going to examine two exemplars used in the self-adaptive systems community.
For each exemplar we will analyze:
- the managed system and the operations available on that system;
- a logging utility to start with a rudimentary form of monitoring; 
- an event scheduler to inject events that simulate the effect of the external environment on the system.

## Environment

Set up the environment importing all the necessary libraries and tools we are going to use today

In [ ]:
import json
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns

In [ ]:
from pysas.system import DeltaIoT, MalwareInfection
from pysas.log import JSONLLogger
from pysas.schedule import DeltaIOTEventsScheduler, MalwareInfectionEventsScheduler

## DeltaIoT

Source: https://people.cs.kuleuven.be/~danny.weyns/software/DeltaIoT/

Reference: https://doi.org/10.1109/SEAMS.2017.21

### Creating instance

Let's start by simply creating an instance of the DeltaIoT network in the original configuration and let's print or visualise someting about its status.

In [ ]:
diot = DeltaIoT.load_from_configs()

Print some general network description

In [ ]:
print(str(diot))

Print some details about the links in the network

In [ ]:
for l in diot.links:
    print(str(l))

Interactive visualization

In [ ]:
fig = diot.plot()

Static visualization

In [ ]:
fig, ax = diot.plot(interactive=False)

### Interacting

#### Nodes

In [ ]:
diot.nodes

In [ ]:
node = diot.get_node(1)
print(str(node))

In [ ]:
node = diot.get_node(7)
print(str(node))

##### Gateways

Get

In [ ]:
gateway = diot.get_gateway(1)
print(str(gateway))

Insert

In [ ]:
from pysas.system.deltaiot.data import Position

diot.insert_gateway(
    params={
        "id": 32,
        "position": Position(600.0, 800.0),
        "view_mote_ids": [15, 8],
        "parent_links": [
            {
                "src_node_id": 15,
                "dest_node_id": 32,
                "power": 15,
                "distribution": 100,
                "snr_eq_params": (0.1, 2.1)
            },
            {
                "src_node_id": 8,
                "dest_node_id": 32,
                "power": 15,
                "distribution": 100,
                "snr_eq_params": (0.2, 2.3)
            }
        ]
    }
)

fig = diot.plot()

Update

In [ ]:
diot.update_gateway(
    id=32,
    params={
        "position":Position(600.0, 900.0),
        "interference":3.0,
    }
)

fig = diot.plot()

Delete

In [ ]:
diot.delete_gateway(32)

fig = diot.plot()

##### Motes

Get

In [ ]:
mote = diot.get_mote(2)
print(str(mote))

Insert

In [ ]:
from pysas.system.deltaiot.data import Position

diot.insert_mote(
    params={
        "id": 64,
        "position": Position(600.0, 800.0),
        "interference": 0.0,
        "battery_capacity": 11880.0,
        "battery_remaining": 11880.0,
        "load": 100,
        "activation_probability": 1.0,
        "last_packet_number": 0,
        "links": [
            {
                "src_node_id": 64,
                "dest_node_id": 8,
                "direction_gateway_id": 1,
                "power": 15,
                "distribution": 100,
                "snr_eq_params": (0.2, 2.0)
            }
        ],
        "parent_links":[
            {
                "src_node_id": 3,
                "dest_node_id": 64,
                "direction_gateway_id": 1,
                "power": 15,
                "distribution": 100,
                "snr_eq_params": (0.1, 2.1)
            }
        ],
        "turn":2
    },
)

fig = diot.plot()

Update

In [ ]:
diot.update_mote(
    id=64,
    params={
        "position":Position(600.0, 900.0),
        "interference":0.0,
        "battery_remaining":200.0,
        "activation_probability":0.5,
        "links": [{
            "src_node_id": 64,
            "dest_node_id": 1,
            "direction_gateway_id": 1,
            "power": 10,
            "distribution": 50,
            "snr_eq_params": (0.3, 1.5)
        }]
    }
)

fig = diot.plot()

Delete

In [ ]:
diot.delete_mote(64)

fig = diot.plot()

#### Link

Get

In [ ]:
link = diot.get_link(12, 7)
print(str(link))

Insert

In [ ]:
diot.insert_link(
    params={
        "src_node_id": 8,
        "dest_node_id": 9,
        "direction_gateway_id": 1,
        "power": 15,
        "distribution": 100,
        "snr_eq_params": (0.2, 2.0)
    }
)

fig = diot.plot()

Update

In [ ]:
diot.update_link(
    params={
        'src_node_id': 8,
        'dest_node_id': 9,
        'direction_gateway_id': 1,
        'power': 10,
        'distribution': 50,
        'snr_eq_params': (0.3, 1.5)
    }
)

fig = diot.plot()

Delete

In [ ]:
diot.delete_link(8, 9)

fig = diot.plot()

#### Turn order

Get

In [ ]:
diot.turn_order

Update

In [ ]:
diot.turn_order = diot.turn_order[::-1]
diot.turn_order

#### Global intereference

Get

In [ ]:
diot.global_interference.get(0)

Update

In [ ]:
diot.global_interference = -4.20
diot.global_interference.get(0)

#### Stepping

In [ ]:
diot.step

In [ ]:
diot.run(num_steps=96)
diot.step

In [ ]:
fig = diot.plot()

### Logging evolution

The simplest way to monitor the evolution of the DeltaIoT system is to attach a logger to it.
First let's create a new istance of your system.

In [ ]:
from datetime import datetime
run_id = datetime.now().strftime('%Y-%m-%d_%H:%M:%S.%f')

diot = DeltaIoT.load_from_configs(path=f'./deltaiot_{run_id}')

Create a logger

In [ ]:
logger = JSONLLogger()

Attach the logger to the managed system.

In [ ]:
diot.logger = logger

When we run a system simulation, the logger will keep track of some metrics describing the evolution of the network.

In [ ]:
diot.run(num_steps=96)

We can read the file.

In [ ]:
with open(f'./deltaiot_{run_id}/logs.jsonl') as f:
    data = [json.loads(line) for line in f]

In [ ]:
data

We can convert the data into a more suitable format

In [ ]:
df = pd.DataFrame.from_dict([elem['system_status'] for elem in data])
df

We can plot the information we collected through the logger.

In [ ]:
metrics = [*df.columns]
labels = [metric.replace('_', ' ').capitalize() for metric in metrics]

figure, axes = plt.subplots(nrows=1, ncols=3, sharex=True, figsize=(18, 4))

for metric, label, ax in zip(metrics, labels, axes):
    ax.plot(df.index, df[metric], c='tab:blue', label='Raw values')
    ax.axhline(df[metric].mean(), xmin=0, xmax=max(df.index), c='tab:red', ls='--', label=f'Mean: {df[metric].mean():.2f}')
    ax.set_xlabel('Step')
    # ax.set_ylabel(ylabel)
    ax.set_title(label)
    ax.grid()
    ax.legend(loc='best')

plt.show()

### Scheduling events

Normally the environment surrounding our system has some effects on it.
We can simulate this by injecting events in our system.
Events can be injected using a scheduler.

First let's create a system with a logger

In [ ]:
run_id = datetime.now().strftime('%Y-%m-%d_%H:%M:%S.%f')
diot = DeltaIoT.load_from_configs(path=f'./deltaiot_with_events_{run_id}')
diot.logger = JSONLLogger()

Create a scheduler with an event to inject in the system.
In this case we are going to inject in the network a mete that floods many messages.

In [ ]:
scheduler = DeltaIOTEventsScheduler.load_from_configs(
    events_config=[{
        'dtype': 'MoteInsertion',
        'step': 50,
        'id': 64,
        'position': (600.0, 800.0),
        'battery_capacity': 11880.0,
        'battery_remaining': 11880.0,
        'load': 100,
        'links': [
            {
                'src_node_id': 64,
                'dest_node_id': 8,
                'direction_gateway_id': 1,
                'power': 15,
                'distribution': 100,
                'snr_eq_params': (0.2, 2.0)
            },
            {
                'src_node_id': 64,
                'dest_node_id': 3,
                'direction_gateway_id': 1,
                'power': 15,
                'distribution': 100,
                'snr_eq_params': (0.2, 2.0)
            },
            {
                'src_node_id': 64,
                'dest_node_id': 12,
                'direction_gateway_id': 1,
                'power': 15,
                'distribution': 100,
                'snr_eq_params': (0.2, 2.0)
            }
        ]
    }]
)

Attach the scheduler to the managed system.

In [ ]:
diot.observer = scheduler

When we run a system simulation, the logger will keep track of some metrics describing the evolution of the network.

In [ ]:
diot.run(num_steps=96)

We can read the file.

In [ ]:
with open(f'./deltaiot_with_events_{run_id}/logs.jsonl') as f:
    data = [json.loads(line) for line in f]

In [ ]:
data

We can convert the data into a more suitable format

In [ ]:
df = pd.DataFrame.from_dict([elem['system_status'] for elem in data if 'system_status' in elem])
df

In [ ]:
[elem for elem in data if 'system_status' not in elem]

We can plot the information we collected through the logger.

In [ ]:
metrics = [*df.columns]
labels = [metric.replace('_', ' ').capitalize() for metric in metrics]

figure, axes = plt.subplots(nrows=1, ncols=3, sharex=True, figsize=(18, 4))

for metric, label, ax in zip(metrics, labels, axes):
    ax.plot(df.index, df[metric], c='tab:blue', label='Raw values')
    ax.axhline(df[metric].mean(), xmin=0, xmax=max(df.index), c='tab:red', ls='--', label=f'Mean: {df[metric].mean():.2f}')
    ax.set_xlabel('Step')
    # ax.set_ylabel(ylabel)
    ax.set_title(label)
    ax.grid()
    ax.legend(loc='best')

plt.show()

## Malware Infection

Affectionately referred to as *[Diego](https://lnu.se/en/staff/diego.perez/)IoT* after its creator.

Source: https://github.com/eres-lnu/antifragile 

Reference: https://doi.org/10.1016/j.jss.2024.112051

### Creating instance

Let's start by simply creating an instance of the Malware Infection network in the original configuration and let's print or visualise someting about its status.

In [ ]:
mi = MalwareInfection.load_from_configs()

Print some general network description

In [ ]:
print(str(mi))

Interactive visualization

In [ ]:
fig = mi.plot(norm=True)

Static visualization

In [ ]:
fig, ax = mi.plot(interactive=False)

### Interacting

##### Software

Get types

In [ ]:
mi.software_type

##### Vulnerabilities

Update

In [ ]:
mi.update_exploit(6, p_attack=0.9)

print(str(mi))

##### Healing time

Get

In [ ]:
mi.mean_healing_time

Set

In [ ]:
mi.mean_healing_time = 10.5
mi.mean_healing_time

##### Nodes

Get

In [ ]:
mi.nodes

##### Edges

Delete

In [ ]:
mi.delete_edge(0, 1)

fig = mi.plot(norm=True)

Restore

In [ ]:
mi.restore_edge(0, 1)

fig = mi.plot(norm=True)

You can only restore edges that where part of the original topology

##### Stepping

In [ ]:
mi.step

In [ ]:
mi.run(num_steps=100)
mi.step

In [ ]:
fig = mi.plot(norm=True)

### Logging evolution

The simplest way to monitor the evolution of the DeltaIoT system is to attach a logger to it.
First let's create a new istance of your system.

In [ ]:
from datetime import datetime

run_id = datetime.now().strftime('%Y-%m-%d_%H:%M:%S.%f')
mi = MalwareInfection.load_from_configs(path=f'./malwareinfection_{run_id}')

Create a logger

In [ ]:
logger = JSONLLogger()

Attach the logger to the managed system.

In [ ]:
mi.logger = logger

When we run a system simulation, the logger will keep track of some metrics describing the evolution of the network.

In [ ]:
mi.run(num_steps=96)

We can read the file.

In [ ]:
with open(f'./malwareinfection_{run_id}/logs.jsonl') as f:
    data = [json.loads(line) for line in f]

In [ ]:
data

We can convert the data into a more suitable format

In [ ]:
df = pd.DataFrame.from_dict([elem['system_status'] for elem in data])
df

We can plot the information we collected through the logger.

In [ ]:
metrics = [*df.columns]
labels = [metric.replace('_', ' ').capitalize() for metric in metrics]

figure, axes = plt.subplots(nrows=1, ncols=2, sharex=True, figsize=(12, 4))

for metric, label, ax in zip(metrics, labels, axes):
    ax.plot(df.index, df[metric], c='tab:blue', label='Raw values')
    ax.axhline(df[metric].mean(), xmin=0, xmax=max(df.index), c='tab:red', ls='--', label=f'Mean: {df[metric].mean():.2f}')
    ax.set_xlabel('Step')
    ax.set_title(label)
    ax.grid()
    ax.legend(loc='best')

plt.show()

### Scheduling events

Normally the environment surrounding our system has some effects on it.
We can simulate this by injecting events in our system.
Events can be injected using a scheduler.

First let's create a system with a logger

In [ ]:
run_id = datetime.now().strftime('%Y-%m-%d_%H:%M:%S.%f')
mi = MalwareInfection.load_from_configs(path=f'./malwareinfection_with_event_{run_id}')
mi.logger = JSONLLogger()

Create a scheduler with an event to inject in the system.
In this case we are going to increase the success probability of some attacks in the network.

In [ ]:
scheduler = MalwareInfectionEventsScheduler.load_from_configs(
    events_config=[{
        'dtype': 'ExploitUpdate',
        'step': 33,
        'exploit_id': 7,
        'p_attack': 0.9
    },{
        'dtype': 'ExploitUpdate',
        'step': 33,
        'exploit_id': 5,
        'p_attack': 0.8
    },{
        'dtype': 'HealingTimeUpdate',
        'step': 33,
        'mean_healing_time': 15.
    }]
)

Attach the scheduler to the managed system.

In [ ]:
mi.observer = scheduler

When we run a system simulation, the logger will keep track of some metrics describing the evolution of the network.

In [ ]:
mi.run(num_steps=100)

We can read the file.

In [ ]:
with open(f'./malwareinfection_with_event_{run_id}/logs.jsonl') as f:
    data = [json.loads(line) for line in f]

In [ ]:
data

We can convert the data into a more suitable format

In [ ]:
df = pd.DataFrame.from_dict([elem['system_status'] for elem in data if 'system_status' in elem])
df

In [ ]:
[elem for elem in data if 'system_status' not in elem]

We can plot the information we collected through the logger.

In [ ]:
metrics = [*df.columns]
labels = [metric.replace('_', ' ').capitalize() for metric in metrics]

figure, axes = plt.subplots(nrows=1, ncols=2, sharex=True, figsize=(12, 4))

for metric, label, ax in zip(metrics, labels, axes):
    ax.plot(df.index, df[metric], c='tab:blue', label='Raw values')
    ax.axhline(df[metric].mean(), xmin=0, xmax=max(df.index), c='tab:red', ls='--', label=f'Mean: {df[metric].mean():.2f}')
    ax.set_xlabel('Step')
    ax.set_title(label)
    ax.grid()
    ax.legend(loc='best')

plt.show()